# Splitting the Dataset

For the beginning, I have decided to go for a 80/10/10 split for train/val/test

## 1.Imports and File Directories

In [11]:
import h5py
import numpy as np

from sklearn.model_selection import train_test_split

DATA_DIR            = '../data/'
DATA_RAW_DIR        = DATA_DIR + 'raw/'
DATA_PROCESSED_DIR  = DATA_DIR + 'processed/'
DATA_SPLITS_DIR     = DATA_DIR + 'splits/'

RMA_CLEAN_DIR       = DATA_PROCESSED_DIR + 'Rma_clean.h5'
UMI_CLEAN_DIR       = DATA_PROCESSED_DIR + 'Umi_clean.h5'
UMA_CLEAN_DIR       = DATA_PROCESSED_DIR + 'Uma_clean.h5'

## 2. Opening the h5 files 

In [12]:
with h5py.File(RMA_CLEAN_DIR, 'r') as f:
    rma_data, rma_mods, rma_snrs = f['Data'][:], f['Mods'][:], f['SNRs'][:]
rma_domain = np.zeros(len(rma_snrs), dtype = np.int8)

with h5py.File(UMI_CLEAN_DIR, 'r') as f:
    umi_data, umi_mods, umi_snrs = f['Data'][:], f['Mods'][:], f['SNRs'][:]
umi_domain = np.ones(len(umi_snrs), dtype = np.int8)

# Pooling all into one dataset
all_data   = np.concatenate([rma_data, umi_data])
all_mods   = np.concatenate([rma_mods, umi_mods])
all_snrs   = np.concatenate([rma_snrs, umi_snrs])
all_domain = np.concatenate([rma_domain, umi_domain])

print(f"Combined dataset: {all_data.shape[0]} rows")

# Free Memory
del rma_data, rma_mods, rma_snrs, umi_data, umi_mods, umi_snrs

Combined dataset: 163840 rows


## 3. Using the Stratify Function

In [13]:
mod_idx = np.argmax(all_mods, axis=1)
all_idx = np.arange(len(mod_idx))

strat_labels = np.array([f"{m}_{s}_{d}" for m, s, d in zip(mod_idx, all_snrs, all_domain)])

train_idx, temp_idx = train_test_split(
    all_idx, train_size=0.8, stratify=strat_labels, random_state=42
)
val_idx, test_idx = train_test_split(
    temp_idx, train_size=0.5, stratify=strat_labels[temp_idx], random_state=42
)

print(f"Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")

Train: 131072  Val: 16384  Test: 16384


## 5. Sanity Check

confirm domain ratio is preserved in each split

In [14]:
for name, idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    domain_ratio = all_domain[idx].mean()
    mod_counts = np.bincount(mod_idx[idx], minlength=5) / len(idx)
    snr_vals, snr_counts = np.unique(all_snrs[idx], return_counts=True)
    snr_frac_range = np.ptp(snr_counts / snr_counts.sum())
    print(f"{name}: domain frac Umi={domain_ratio:.3f} (~0.5) | "
          f"mod frac range={mod_counts.min():.3f}-{mod_counts.max():.3f} (~0.2 each) | "
          f"snr frac spread={snr_frac_range:.4f} (should be ~0)")

train: domain frac Umi=0.500 (~0.5) | mod frac range=0.200-0.200 (~0.2 each) | snr frac spread=0.0000 (should be ~0)
val: domain frac Umi=0.500 (~0.5) | mod frac range=0.200-0.200 (~0.2 each) | snr frac spread=0.0002 (should be ~0)
test: domain frac Umi=0.500 (~0.5) | mod frac range=0.200-0.200 (~0.2 each) | snr frac spread=0.0003 (should be ~0)


## 6. Saving the Files

In [15]:
def save_split(path, idx):
    with h5py.File(path, 'w') as f:
        f.create_dataset('Data',   data=all_data[idx])
        f.create_dataset('Mods',   data=all_mods[idx])
        f.create_dataset('SNRs',   data=all_snrs[idx])
        f.create_dataset('Domain', data=all_domain[idx])  # 0 = Rma, 1 = Umi

In [16]:
save_split(DATA_SPLITS_DIR + 'train.h5', train_idx)
save_split(DATA_SPLITS_DIR + 'val.h5',   val_idx)
save_split(DATA_SPLITS_DIR + 'test.h5',  test_idx)

print("Saved train.h5, val.h5, test.h5 to", DATA_SPLITS_DIR)

Saved train.h5, val.h5, test.h5 to ../data/splits/


## 7. Second Dataset: RMa + UMi + UMa

In [17]:
with h5py.File(RMA_CLEAN_DIR, 'r') as f:
    rma_data, rma_mods, rma_snrs = f['Data'][:], f['Mods'][:], f['SNRs'][:]
rma_domain_3 = np.zeros(len(rma_snrs), dtype=np.int8)          # 0 = Rma

with h5py.File(UMI_CLEAN_DIR, 'r') as f:
    umi_data, umi_mods, umi_snrs = f['Data'][:], f['Mods'][:], f['SNRs'][:]
umi_domain_3 = np.ones(len(umi_snrs), dtype=np.int8)           # 1 = Umi

with h5py.File(UMA_CLEAN_DIR, 'r') as f:
    uma_data, uma_mods, uma_snrs = f['Data'][:], f['Mods'][:], f['SNRs'][:]
uma_domain_3 = np.full(len(uma_snrs), 2, dtype=np.int8)        # 2 = Uma

all_data_full   = np.concatenate([rma_data, umi_data, uma_data])
all_mods_full   = np.concatenate([rma_mods, umi_mods, uma_mods])
all_snrs_full   = np.concatenate([rma_snrs, umi_snrs, uma_snrs])
all_domain_full = np.concatenate([rma_domain_3, umi_domain_3, uma_domain_3])

print(f"Combined 3-domain dataset: {all_data_full.shape[0]} rows")

del rma_data, rma_mods, rma_snrs, umi_data, umi_mods, umi_snrs, uma_data, uma_mods, uma_snrs

Combined 3-domain dataset: 245760 rows


### 7.1. Stratify w sklearn

In [18]:
mod_idx_full = np.argmax(all_mods_full, axis=1)
all_idx_full = np.arange(len(mod_idx_full))

strat_labels_full = np.array([
    f"{m}_{s}_{d}" for m, s, d in zip(mod_idx_full, all_snrs_full, all_domain_full)
])

train_idx_full, temp_idx_full = train_test_split(
    all_idx_full, train_size=0.8, stratify=strat_labels_full, random_state=42
)
val_idx_full, test_idx_full = train_test_split(
    temp_idx_full, train_size=0.5,
    stratify=strat_labels_full[temp_idx_full], random_state=42
)

print(f"Train: {len(train_idx_full)}  Val: {len(val_idx_full)}  Test: {len(test_idx_full)}")

Train: 196608  Val: 24576  Test: 24576


### 7.2. Sanity Check

In [19]:
for name, idx in [('train', train_idx_full), ('val', val_idx_full), ('test', test_idx_full)]:
    vals, counts = np.unique(all_domain_full[idx], return_counts=True)
    fracs = counts / counts.sum()
    breakdown = ", ".join(f"domain {v}={f:.3f}" for v, f in zip(vals, fracs))
    print(f"{name}: {breakdown}  (should be ~0.333 each)")


train: domain 0=0.333, domain 1=0.333, domain 2=0.333  (should be ~0.333 each)
val: domain 0=0.333, domain 1=0.333, domain 2=0.333  (should be ~0.333 each)
test: domain 0=0.333, domain 1=0.333, domain 2=0.333  (should be ~0.333 each)


### 7.3. Saving the 3 domain files

In [20]:
import os

DATA_SPLITS_3DOM_DIR = DATA_SPLITS_DIR + 'rma_umi_uma/'

os.makedirs(DATA_SPLITS_3DOM_DIR, exist_ok=True)

def save_split(path, data, mods, snrs, domain, idx):
    with h5py.File(path, 'w') as f:
        f.create_dataset('Data',   data=data[idx])
        f.create_dataset('Mods',   data=mods[idx])
        f.create_dataset('SNRs',   data=snrs[idx])
        f.create_dataset('Domain', data=domain[idx])  # 0 = Rma, 1 = Umi, 2 = Uma

save_split(DATA_SPLITS_3DOM_DIR + 'train.h5', all_data_full, all_mods_full, all_snrs_full, all_domain_full, train_idx_full)
save_split(DATA_SPLITS_3DOM_DIR + 'val.h5',   all_data_full, all_mods_full, all_snrs_full, all_domain_full, val_idx_full)
save_split(DATA_SPLITS_3DOM_DIR + 'test.h5',  all_data_full, all_mods_full, all_snrs_full, all_domain_full, test_idx_full)

print("Saved train.h5, val.h5, test.h5 to", DATA_SPLITS_3DOM_DIR)

Saved train.h5, val.h5, test.h5 to ../data/splits/rma_umi_uma/
